# Eclipsing binaries (EBs)

This notebook can be used to have a quicklook at the that product for the simulations.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# PlatoSim libraries
import platosim.plot      as pt
import platosim.mocka     as mk
import platosim.utilities as ut
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_notebook
setup_notebook()

from IPython.display import display, HTML
display(HTML("<style>.container {width:70% !important; }</style>"))

In [ ]:
path = os.getenv('PLATO_WORKDIR') + 'binary'
pdir = f'{path}/plots'
idir = f'{path}/input'
vdir = f'{path}/varsource'
sdir = f'{path}/simulations'

---
## 1. Stellar catalogue
---

In [ ]:
# Load PLATO-CS catalogue
df = pd.read_feather(f'{path}/../mocka/input/starcat_GaiaDR3_PlatoCS.ftr')
# df = pd.read_feather(f'{path}/../mocka/input/starcat_GaiaDR3_PlatoCS.ftr')
df.gaiaDR3 = df.gaiaDR3.astype(float)
df.head()

In [ ]:
# Load Luc's binary table
dt = pd.read_csv(f'{path}/table_OBAF_EB_catalogue.csv')
dt.head()

In [ ]:
fig, ax = pt.drawStarsInSkyAitoff(dt.ra, dt.dec, figsize=(10,6))

In [ ]:
# Cross-match catalogue
dt0 = dt[dt['GAIA_DR3'].isin(df['gaiaDR3'])].sort_values(by=['GAIA_DR3']).reset_index(drop=True)
df0 = df[df['gaiaDR3'].isin(dt['GAIA_DR3'])].sort_values(by=['gaiaDR3']).reset_index(drop=True)
# dt0 = dt[dt['GAIA_DR3'].isin(df['source_gaia_dr3'])].sort_values(by=['GAIA_DR3']).reset_index(drop=True)
# df0 = df[df['source_gaia_dr3'].isin(dt['GAIA_DR3'])].sort_values(by=['source_gaia_dr3']).reset_index(drop=True)

In [ ]:
fig, ax = pt.drawStarsInSkyAitoff(df0.ra, df0.dec, df0.Gmag, figsize=(10,7),
                                  title=f'Total number of stars : {df0.shape[0]} of {dt.shape[0]}')

In [ ]:
# Select catalogue EBs
nstars = int(295/4)

# All stars
ds = df0
n_all = [ds[ds.ncams == i].shape[0] for i in [6,12,18,24]]
print('Before cuts:', n_all)

# Perform cuts
ds = ds[(ds.Pmag > 8) & (ds.Pmag < 16)]
n_all = [ds[ds.ncams == i].shape[0] for i in [6,12,18,24]]
print('After  cuts:', n_all)

# Fetch sample
ds = mk.fetch_sample(ds, nstars, seed=None, max_mag=14, max_weight=1)

In [ ]:
fig, ax = pt.plotPlatoFOV('LOPS2', raStars=ds.ra, decStars=ds.dec, c=ds.Pmag, s=70, lw=0.3, 
                          ncamStars=True, clabel=r'$\mathcal{P}$ [mag]', figsize=(9,9))
fig.savefig(f'{pdir}/starcat_sky_EBs.png', bbox_inches='tight', dpi=200)

In [ ]:
# Create target catalogue
ds.reset_index(drop=True, inplace=True)
ds.to_feather(f'{idir}/starcat_GaiaDR3_PlatoCS_EB_targets.ftr')

In [ ]:
# Create contaminant catalogue
dc = ut.getContaminants(ds, df, column='gaiaDR3')
dc.to_feather(f'{idir}/starcat_GaiaDR3_PlatoCS_EB_contaminants.ftr')

In [ ]:
# # Small script to move varsource files into a sub-directory 
# files = sorted(glob.glob(f'{vdir}/varsource_*'))
# for i in range(len(files)):
#     f = Path(files[i])
#     folder = Path(vdir) / f.stem[-9:]
#     folder.mkdir(parents=True, exist_ok=True)
#     os.system(f'mv {str(f)} {str(folder)}')

---
## 2. Post-processing
---

In [ ]:
# Load file to test detrending with
filename = f"{sdir}/test_hdf5/000000001/000000001_Ncam2.1_Q1.hdf5"; pfile = f'{pdir}/test_detrending_000000001_Ncam2.1_Q1.png'
filename = f"{sdir}/test_hdf5/000000080/000000080_Ncam1.1_Q1.hdf5"; pfile = f'{pdir}/test_detrending_000000080_Ncam1.1_Q1.png'
filename = f"{sdir}/test_hdf5/000000160/000000160_Ncam1.1_Q1.hdf5"; pfile = f'{pdir}/test_detrending_000000160_Ncam1.1_Q1.png'

# Load a single mission quarter light curve
lc = LightCurve(filename, mode="single")
dt = lc.star()
dt

In [ ]:
# Show a quarter light curve
lc.plot(flux_unit='ppt', median_filter=1, input_model=False, figsize=(9,5));

### 2.1. Test detrending

In [ ]:
# Generate plot to save
df = lc.detrend(model='lowess', lowess_frac=1/5, segments=True, replace=False, plot=False)
fig, ax = lc.plot_detrend(df, figsize=(9,12))
fig.savefig(pfile, bbox_inches='tight', dpi=300)

In [ ]:
# Perform detrending that can be used for clipping after
df = lc.detrend(model='lowess', lowess_frac=1/5, segments=True, replace=True, plot=True)

### 2.2. Test outlier rejection

In [ ]:
# Remove outliers (due to photon noise and cosmic ray hits)
df = lc.clip(model='wotan', sigma_lower=6, sigma_upper=3, flux_unit='ppt', plot=True)

---
## 3. Analysis of simulations
---

In [ ]:
# In multi mode we parse the entire directory of files
star = '000000002'
lcs = LightCurve(f'{path}/{star}', mode="multi")

In [ ]:
lcs = LightCurve(f'{path}', mode="multi")

In [ ]:
lcs.unpack()

### *Use multi-mode as single-mode*

In [ ]:
# Fetch files in folder
filenames = lcs.files(suffix='ftr')
filenames[:3]

In [ ]:
# Again one can fetch the first light curve for this star
lc = LightCurve(filenames[0])
lc.data().head()

### *Simulation statistics*

The `.table` files each contain a small overview of the specific simulation. It is much handier to have a single file to search information from, hence, we can merge to one single overview table as follows. It possible to remove the redundant `.table` files during the process using `clean=True`:

In [ ]:
df = lcs.stat_sim_table(ofile=f'{path}/../table/lc_{star}.tab', clean=True)
df.head()

### *View simulations*

In [ ]:
fig, ax = lcs.plot_multi(suffix='ftr', group=1, camera=1, quarter=False, 
                         flux_median=144, alpha=0.1, figsize=(9,5))

In [ ]:
lc = lcs.merge(suffix='ftr', binsize=1/6, flux_group_mean=True, flux_offset=True, 
               ofile=f'{path}/../final/lc_{star}.ftr')

In [ ]:
lc.plot(input_model=True, flux_unit='ppm', median_filter=1, alpha=0.1, figsize=(9,6));

---
## Final data products
---

In [ ]:
path = '/lhome/nicholas/software/workdir/cs-binary/finals'
star = 'mag085/lc_000000001.ftr'

# In multi mode we parse the entire directory of files
lc = LightCurve(f'{path}/{star}', mode="final")

# Introduce gaps from file
inputFileGap = '/lhome/nicholas/software/workdir/mocka/input/instrumentGAP.tab'
lc.gaps(inputFileGap, replace=True, plot=False);

# Show a quarter light curve
lc.plot(flux_unit='ppm', median_filter=1, input_model=True, legend=False, figsize=(9,5));

In [ ]:
path = '/lhome/nicholas/software/workdir/cs-binary/finals'
star = 'mag085/lc_000000090.ftr'

# In multi mode we parse the entire directory of files
lc = LightCurve(f'{path}/{star}', mode="final")

# Introduce gaps from file
inputFileGap = '/lhome/nicholas/software/workdir/mocka/input/instrumentGAP.tab'
lc.gaps(inputFileGap, replace=True, plot=False);

# Show a quarter light curve
lc.plot(flux_unit='ppt', median_filter=1, input_model=False, legend=False, figsize=(9,5));